# MTL — Colab/Kaggle GPU training

Run cells top to bottom. This trains the shared ResNet50+FPN backbone
jointly on detection (RetinaNet) + semantic segmentation (FCN) +
multi-label classification, on a COCO subset.


## 1. Install dependencies
Colab usually ships a CUDA-matched torch/torchvision already — only reinstall if missing.

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())

# Uncomment only if the above shows no CUDA:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

!pip install pycocotools PyYAML tqdm scikit-learn requests

## 2. Get the repo
Either clone from git, or upload a zip of this project via the Colab file browser and unzip it.

In [ ]:
# Option A: git clone
# !git clone <your-repo-url> mtl

# Option B: unzip an uploaded mtl.zip
# !unzip -q mtl.zip -d .

%cd mtl
!pip install -e .

## 3. Build the COCO subset
Download full COCO annotations/images first (e.g. via the COCO website or Kaggle's COCO dataset), then filter down to the training subset.

In [ ]:
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q annotations_trainval2017.zip

!python scripts/prepare_coco_subset.py \
    --ann-file annotations/instances_train2017.json \
    --out data/coco_subset/annotations/instances_train_subset.json \
    --n-images 22500
!python scripts/prepare_coco_subset.py \
    --ann-file annotations/instances_val2017.json \
    --out data/coco_subset/annotations/instances_val_subset.json \
    --n-images 2000

# Download only the subset's images (via each image's coco_url) - far
# cheaper than the full train2017.zip (~18GB) / val2017.zip (~1GB).
!python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_train_subset.json \
    --out-dir data/coco_subset/images/train
!python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_val_subset.json \
    --out-dir data/coco_subset/images/val

## 4. Train

In [ ]:
!python scripts/train.py --config configs/train_colab_gpu.yaml

## 5. Evaluate + visualize a few predictions

In [ ]:
!python scripts/eval.py --config configs/train_colab_gpu.yaml --checkpoint checkpoints/colab_gpu_epoch15.pt

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision.utils import draw_bounding_boxes

from mtl.config import load_config
from mtl.datasets.coco_multitask import CocoMultiTaskDataset
from mtl.engine.checkpoint import load_checkpoint
from mtl.models.multitask_model import MultiTaskModel
from mtl.utils.device import resolve_device

cfg = load_config("configs/train_colab_gpu.yaml")
device = resolve_device(cfg.train.device)
dataset = CocoMultiTaskDataset(cfg.data.val_ann_file, cfg.data.val_img_dir, img_size=cfg.data.img_size, train=False)

model = MultiTaskModel(
    det_num_classes=dataset.num_classes,
    seg_num_classes=dataset.num_classes + 1,
    cls_num_labels=dataset.num_classes,
).to(device)
load_checkpoint(model, optimizer=None, path="checkpoints/colab_gpu_epoch15.pt", map_location=str(device))
model.eval()

image, target = dataset[0]
with torch.no_grad():
    out = model(image.unsqueeze(0).to(device))

det = out["detections"][0]
keep = det["scores"] > 0.5
img_uint8 = ((image * 0.5 + 0.5) * 255).clamp(0, 255).byte()
drawn = draw_bounding_boxes(img_uint8, det["boxes"][keep].cpu())
plt.imshow(drawn.permute(1, 2, 0))
plt.title(f"top labels: {out['cls_pred'][0].topk(5).indices.tolist()}")
plt.show()